# Vending Machine Assistant - CPU Inference Demo
Use this notebook to test the fine-tuned `Qwen2.5-1.5B-Instruct` model for Joint Intent Recognition and Slot Filling **purely on CPU hardware**.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Change working directory to your project folder
%cd /content/drive/MyDrive/Voice_Interaction_For_Robot

In [ ]:
# Install dependencies (CPU friendly, NO bitsandbytes needed)
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate psutil

In [ ]:
import torch
import json
import time
import psutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading Base Model on CPU (this may take a minute depending on RAM)...")
STUDENT_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER_PATH = './student_1.5b_adapter'

t0 = time.time()
# Load in float32 for maximum CPU compatibility
base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, 
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Applying LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

load_time = time.time() - t0
mem_mb = psutil.Process().memory_info().rss / (1024 * 1024)

print(f"\n✅ Model loaded successfully on CPU!")
print(f"⏱️ Load time: {load_time:.1f} seconds")
print(f"💾 RAM usage: {mem_mb:.0f} MB")

In [ ]:
SYSTEM_PROMPT = """You are a Vietnamese Voice-Enabled Vending Machine Assistant.
Your task is Joint Intent Recognition and Entity Extraction (Slot Filling).
Extract the user's intent and product items into a structured JSON format.

Supported intents: greeting, show_menu, buy_product, add_product, change_product, payment, cancel, help, unknown
Canonical products: coca, pepsi, sting, aquafina, 7up (normalize all aliases)

Output JSON Format EXACTLY like this:
{
  "intent": "buy_product",
  "items": [
    {
      "product": "coca",
      "quantity": 2
    }
  ]
}

DO NOT output any explanation or markdown, just the raw JSON."""

def predict(text):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Text: \"{text}\""}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    inf_time = time.time() - t0
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    cleaned = response.strip()
    if "```json" in cleaned: cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```" in cleaned: cleaned = cleaned.split("```")[1].split("```")[0].strip()
    
    try:
        parsed_json = json.loads(cleaned)
        return parsed_json, inf_time
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw_output": response}, inf_time

## 🤖 Try it yourself!

In [ ]:
tests = [
    "cho tôi 2 lon bép si và 1 chai cô ca nhé",
    "à thôi không lấy bép si nữa",
    "đổi sang 7up rồi thanh toán luôn giùm",
    "menu có những món gì vậy",
    "cho thêm 3 chai nước suối a qua phi na nha"
]

# Warmup run to initialize CPU caches
predict("warmup")

for t in tests:
    print(f"\n🗣️ User: {t}")
    result, lat = predict(t)
    print(f"⏱️ Latency: {lat:.3f}s")
    print(f"🤖 Assistant:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# Interactive Testing Loop
print("Enter a sentence to test (or type 'quit' to exit):")
while True:
    user_input = input("🗣️ User: ")
    if user_input.strip().lower() == 'quit':
        break
    
    result, lat = predict(user_input)
    print(f"⏱️ Latency: {lat:.3f}s")
    print("🤖 Assistant:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print("-" * 50)